In [ ]:
import os
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms.v2 as transforms
from torchvision import datasets
from sklearn.model_selection import StratifiedShuffleSplit
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from torchvision.models import vgg16
from torchvision.models import VGG16_Weights

device = "cpu"
print(f'Usando: {device}')

# EXPERMENTO 0

Antes de comenzar a congelar layers del VGG-16, definimos la arquitectura del modelo y la emprimimos para poder analizarla. Además de haber cambiado el layer del output (6 clases en vez de 1000, como venía predefenido en el modelo original), cambiamos el layer 15° donde hay out_features = 500, en vez de 4096 como en el modelo original. Esto fue un cambio hecho por el profesor, probablmente porque tenemos un problema más sencillo: 6 clases en vez de 1000.

In [ ]:
#weights = VGG16_Weights.FIXME
weights = VGG16_Weights.DEFAULT
vgg_model = vgg16(weights=weights)

N_CLASSES = 6

my_model = nn.Sequential(
    vgg_model.features,
    vgg_model.avgpool,
    nn.Flatten(),
    vgg_model.classifier[0:3],
    nn.Linear(4096, 500),
    nn.ReLU(),
    nn.Linear(500, N_CLASSES)
)
my_model

## Funciones de entrenamiento, validación y test.

La función de test genera un archivo csv que nos permite validar el label predicho y el label correcto.

In [ ]:
def train(model, loader, N, optimizer, loss_fn, device):
    model.train()
    total_loss, correct = 0, 0
    for x, y in tqdm(loader, desc='  train', leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out  = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += out.argmax(1).eq(y).sum().item()
    print(f'  Train | Loss: {total_loss/len(loader):.4f} | Acc: {correct/N:.4f}')
    return correct / N


def validate(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out        = model(x)
            total_loss += loss_fn(out, y).item()
            correct    += out.argmax(1).eq(y).sum().item()
            total      += y.size(0)
    print(f'  Val   | Loss: {total_loss/len(loader):.4f} | Acc: {correct/total:.4f}')
    return correct / total


def test(model, loader, loss_fn, device, csv_path='test_predictions.csv'):
    import csv

    model.eval()
    total_loss, correct, total = 0, 0, 0
    rows = []

    dataset = loader.dataset
    if isinstance(dataset, Subset):
        base_dataset = dataset.dataset
        ordered_indices = list(dataset.indices)
    else:
        base_dataset = dataset
        ordered_indices = list(range(len(dataset)))

    class_names = getattr(base_dataset, 'classes', None)
    start = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            pred = out.argmax(1)

            total_loss += loss_fn(out, y).item()
            correct += pred.eq(y).sum().item()
            bsz = y.size(0)
            total += bsz

            batch_indices = ordered_indices[start:start + bsz]
            start += bsz

            for j, base_idx in enumerate(batch_indices):
                img_path, _ = base_dataset.samples[base_idx]
                image_id = os.path.splitext(os.path.basename(img_path))[0]
                true_idx = y[j].item()
                pred_idx = pred[j].item()

                true_label = class_names[true_idx] if class_names is not None else true_idx
                pred_label = class_names[pred_idx] if class_names is not None else pred_idx
                rows.append([image_id, true_label, pred_label])

    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['image_id', 'true_label', 'pred_label'])
        writer.writerows(rows)

    acc = correct / total
    print(f'  Test  | Loss: {total_loss/len(loader):.4f} | Acc: {acc:.4f}')
    print(f'  CSV guardado en: {csv_path}')
    return acc

## Función de perdida y optimizador

In [ ]:

loss_function = nn.CrossEntropyLoss()
optimizer = Adam(filter(lambda p: p.requires_grad, my_model.parameters()))
my_model = my_model.to(device)

## Definición de transformaciones para data loaders

In [ ]:
IMG_WIDTH, IMG_HEIGHT = (224, 224)

train_tf = transforms.Compose([
    transforms.Resize((IMG_WIDTH, IMG_HEIGHT)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_WIDTH, IMG_HEIGHT)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

## Obtener data sets

In [ ]:
# Load data
base_path = os.path.join(os.path.dirname(os.path.abspath('quiz_2.ipynb')), 'data')

# Intel dataset has one extra nested folder level for train/test.
train_path = os.path.join(base_path, 'seg_train', 'seg_train')
val_path   = os.path.join(base_path, 'seg_test', 'seg_test')

train_dataset_full = datasets.ImageFolder(root=train_path, transform=train_tf)
val_dataset_full   = datasets.ImageFolder(root=val_path,   transform=val_tf)

print(f'Clases detectadas: {train_dataset_full.classes}')
print(f'Total train: {len(train_dataset_full):,} imágenes')
print(f'Total val:   {len(val_dataset_full):,} imágenes')

if len(train_dataset_full.classes) <= 1:
    raise ValueError('Ruta de dataset incorrecta: se detecto 1 sola clase.')

## Dividir validación y test

In [ ]:
targets     = np.array(val_dataset_full.targets)
sample_size = 1500

splitter = StratifiedShuffleSplit(n_splits=2, test_size=sample_size, random_state=42)
sample_datasets = []
for _, sample_idx in splitter.split(np.zeros(len(targets)), targets):
    sample_datasets.append(Subset(val_dataset_full, sample_idx))

val_dataset  = sample_datasets[0]
test_dataset = sample_datasets[1]

print(f'Dataset de validación:     {len(val_dataset)} imágenes')
print(f'Dataset de prueba interno: {len(test_dataset)} imágenes')

## Definir data loaders

In [ ]:
torch.backends.cudnn.benchmark = True
nw = min(4, os.cpu_count())

train_loader = DataLoader(train_dataset_full, batch_size=50, shuffle=True,
                          num_workers=nw, pin_memory=True, persistent_workers=True, prefetch_factor=2)
valid_loader = DataLoader(val_dataset,        batch_size=50, shuffle=False,
                          num_workers=nw, pin_memory=True, persistent_workers=True, prefetch_factor=2)
test_loader  = DataLoader(test_dataset,       batch_size=50, shuffle=False,
                          num_workers=nw, pin_memory=True, persistent_workers=True, prefetch_factor=2)

train_N = len(train_loader.dataset)
valid_N = len(valid_loader.dataset)
test_N  = len(test_loader.dataset)

for images, labels in train_loader:
    print(f'Batch shape: {images.shape}')  # Debe ser [64, 3, 224, 224]
    break

# EXPERIMENTO 1
Vamos a entrenar únicamente el último layer del modelo, manteniendo congelados los
demás layers.

In [ ]:
# congelar todo
for p in my_model.parameters():
    p.requires_grad = False

# descongelar solo la ultima capa
for p in my_model[-1].parameters():
    p.requires_grad = True

total     = sum(p.numel() for p in my_model.parameters())
trainable = sum(p.numel() for p in my_model.parameters() if p.requires_grad)
print(f'Parámetros totales:      {total:,}')
print(f'Parámetros entrenables:  {trainable:,}')
print(f'Parámetros congelados:   {total - trainable:,}')
my_model

In [10]:
epochs = 1

train_accs_exp1 = []
val_accs_exp1   = []
best_val_exp1   = 0.0

print('=== EXPERIMENTO 1: Solo última capa (modelo congelado) ===')
for epoch in range(epochs):
    print('Epoch: {}'.format(epoch))
    tr_acc  = train(my_model, train_loader, train_N, optimizer, loss_function, device)
    val_acc = validate(my_model, valid_loader, loss_function, device)
    train_accs_exp1.append(tr_acc)
    val_accs_exp1.append(val_acc)
    if val_acc > best_val_exp1:
        best_val_exp1 = val_acc
        torch.save(my_model.state_dict(), 'best_exp1.pth')

print(f'\nMejor Val Acc Experimento 1: {best_val_exp1:.4f}')
my_model.load_state_dict(torch.load('best_exp1.pth', weights_only=True))

test_acc_exp1 = test(my_model, test_loader, loss_function, device, csv_path='test_predictions_exp1.csv')

KeyboardInterrupt: 